# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Access metadata as an object, not a dictionary
print(f"{dataset.metadata.name}: {dataset.metadata.description}\n")
print(f"Published: {dataset.metadata.date_published}")
print(f"Identifier: {dataset.metadata.identifier}")
print(f"Version: {dataset.metadata.version}")
print(f"License: {dataset.metadata.license}")


## 2. Data Overview
Review available record sets, fields, and their IDs.

Below, we list all available Record Sets, their `@id`s, and their fields (features/columns) as described by the schema.

We will reference everything by `@id`, as required by the Croissant standard.

In [ ]:
# Helper function to print information for each record set
from pprint import pprint
def print_record_sets(ds):
    print("Record Sets present in the dataset:")
    for rs in ds.metadata.record_sets:
        print(f"- @id: {rs.id}")
        print(f"  Name: {getattr(rs, 'name', 'N/A')}")
        print(f"  Description: {getattr(rs, 'description', 'N/A')}")
        print(f"  Fields (by @id):")
        for field in rs.fields:
            print(f"    - @id: {field.id} | Name: {getattr(field, 'name', 'N/A')}")
        print()

print_record_sets(dataset)


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

All entities are referenced by their `@id` fields as required. Update the variables below if you want to select a different record set or field.

In [ ]:
# Choose which record sets to extract. Use the @id string(s) from the previous cell (printout), example shown:
record_sets_to_extract = [
    # Example: 'cr:recordset/proteins'
]

# If there is only one, you may use something like:
# [rs.id for rs in dataset.metadata.record_sets]
if not record_sets_to_extract:
    # If no manual selection, extract all
    record_sets_to_extract = [rs.id for rs in dataset.metadata.record_sets]

dataframes = {}  # @id -> DataFrame
for record_set_id in record_sets_to_extract:
    print(f"Loading records from Record Set @id: {record_set_id}")
    # Returns iterable of dicts
    records = list(dataset.records(record_set=record_set_id))
    if len(records) == 0:
        print("-- No records found in this set.")
    else:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"-- Columns: {df.columns.tolist()}")
        display(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering records, normalizing numeric fields, categorizing, and grouping.

Replace `<numeric_field_id>` and `<group_field_id>` with the actual column names as listed above. All columns correspond to their Croissant field `@id`. You may need to look up in the previous cell which columns are available.


In [ ]:
# Choose the primary record set (use one from above -- typically the main, largest table)
if len(dataframes) > 0:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    print(f"Will perform EDA on record set: {record_set_id}")
    print(df.info())
    
    # --- SELECT FIELD BY @id ---
    # Replace this with the field/column name or @id string for your main numeric analysis.
    # For example, if there is a column '@id': 'protein_abundance', use that.
    numeric_field = ''  # e.g., 'protein_abundance'
    possible_numeric = df.select_dtypes(include=['number']).columns.tolist()
    if not numeric_field and len(possible_numeric) > 0:
        numeric_field = possible_numeric[0]
        print(f"Auto-selected numeric field: {numeric_field}")
    elif not numeric_field:
        print("No numeric fields found; select one manually from column list.")

    threshold = 10
    if numeric_field:
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        display(filtered_df.head())
        norm_col = f"{numeric_field}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, norm_col]].head())

        # Grouping and mean aggregation (e.g., by protein description or other categorical field):
        group_fields = df.select_dtypes(include=["object"]).columns.tolist()
        group_field = ''  # Fill in if you want, or auto-select first string field
        if not group_field and len(group_fields) > 0:
            group_field = group_fields[0]
            print(f"Auto-selected group field: {group_field}")
        if group_field:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped mean of numeric fields by '{group_field}':")
            display(grouped_df.head())
else:
    print("No DataFrames loaded; check data extraction above.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. The example below generates a histogram and a scatter plot for numeric fields.


In [ ]:
# Visualization examples
import matplotlib.pyplot as plt

if len(dataframes) > 0:
    df = list(dataframes.values())[0]
    # Histogram for the selected numeric field
    if 'numeric_field' in locals() and numeric_field and numeric_field in df.columns:
        plt.figure(figsize=(6,4))
        df[numeric_field].hist(bins=30)
        plt.title(f"Distribution of {numeric_field}")
        plt.xlabel(numeric_field)
        plt.ylabel('Frequency')
        plt.show()

    # If two or more numeric columns, plot scatter
    numeric_columns = df.select_dtypes(include=['number']).columns.tolist()
    if len(numeric_columns) > 1:
        plt.figure(figsize=(6,4))
        plt.scatter(df[numeric_columns[0]], df[numeric_columns[1]], alpha=0.6)
        plt.xlabel(numeric_columns[0])
        plt.ylabel(numeric_columns[1])
        plt.title(f"Scatter plot: {numeric_columns[0]} vs {numeric_columns[1]}")
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- This notebook demonstrated how to load a Croissant-compliant dataset using `mlcroissant`, inspect its metadata, view available record sets, and load and process tabular data using record set and field `@id`s.
- You are encouraged to adjust field/filter/group logic depending on your particular scientific questions and the dataset's structure.